# Reconstrucción de la DSA desde el EEG crudo

## Objetivo y alcance

El objetivo es obtener, a partir del EEG crudo, una matriz tiempo-frecuencia
comparable con la DSA almacenada en el archivo `.f_a`, respetando:

- la línea temporal oficial del `.spa`;
- una resolución temporal de un segundo;
- frecuencias entre 0,5 y 30 Hz en intervalos de 0,5 Hz;
- la escala de potencia utilizada para comparar y visualizar ambas matrices.

No se pretende reproducir exactamente el algoritmo interno del BIS, que es
propietario y no está completamente documentado. Tampoco se reconstruye el
índice BIS ni se afirma que el fabricante utilice exactamente todas las
operaciones implementadas. Se desarrolla una aproximación reproducible y se
valida frente al `.f_a`.

# Archivos y selección de canales

Los archivos utilizados son:

| Archivo | Finalidad |
|:---|:---|
| `.r2a` | EEG crudo de una monitorización unilateral |
| `.r4a` | EEG crudo de una monitorización bilateral |
| `.h_a` | Número de canales, frecuencia de muestreo, pendiente y offset |
| `.t_a` | Instante inicial del EEG crudo |
| `.spa` | Línea temporal oficial y variables procesadas |
| `.f_a` | DSA exportada por el monitor, utilizada como referencia |

La extensión de la señal cruda permite detectar automáticamente la modalidad.

- **Unilateral:** se reconstruye una matriz a partir del canal 1.
- **Bilateral:** se reconstruye la matriz izquierda con el canal 1 y la
  derecha con el canal 3.

No se emplea `(C1-C2)/2` ni se promedian las potencias C1-C2 y C3-C4 en la
implementación final. Esta elección sigue la correspondencia entre los
datasets espectrales exportados y los canales 1 y 3 indicada para los
registros VISTA de cuatro canales.

# Lectura y conversión de la señal

La señal cruda se almacena como enteros con signo de 16 bits en formato
little-endian. Los valores no deben interpretarse directamente como
microvoltios.

La cabecera `.h_a` proporciona la frecuencia de muestreo y los coeficientes de
calibración. Para cada muestra:

$$
x_{\mu V}=x_{\mathrm{digital}}\cdot m+b,
$$

donde \(m\) es la pendiente y \(b\) el offset extraídos de la cabecera.

No se utiliza un factor fijo como `0.0511`, porque la conversión debe respetar
los parámetros del registro concreto. En los datos evaluados, la frecuencia
de muestreo es 128 Hz, es decir, se registran 128 muestras por canal y segundo.

# Alineación temporal

El archivo `.t_a` proporciona el comienzo de la señal cruda y el `.spa`
define la línea temporal oficial de la sesión. El `.spa` manda sobre la
longitud final.

1. Se calcula el desfase entre el inicio del raw y el primer segundo del
   `.spa`.
2. Si el raw comienza antes, se eliminan las muestras iniciales sobrantes.
3. Si comienza después, se introducen valores ausentes al principio.
4. El final se recorta o completa hasta obtener exactamente
   `n_segundos_spa × fs` muestras.

Este paso evita asumir que el primer byte del archivo crudo coincide siempre
con el comienzo de las variables procesadas.

# Filtro pasa-altos causal

Antes del análisis espectral se aplica un filtro Butterworth pasa-altos causal
de primer orden. La frecuencia de corte se obtiene del campo `LoFilter`:

| Código | Corte |
|---:|---:|
| 0 | 0,25 Hz |
| 1 | 1 Hz |
| 2 | 2 Hz |
| 3 | 2,5 Hz |

El filtro se aplica sobre la señal temporal, antes de Welch. Un filtro
pasa-altos pretende atenuar componentes lentas y desplazamientos de línea
base; no debe aplicarse después de la FFT como si se tratase de corregir una
imagen ya construida.

La implementación utiliza `sosfilt`, que solo depende de muestras presentes y
pasadas. Cada bloque continuo de muestras válidas se filtra por separado para
no propagar el estado del filtro a través de huecos con `NaN`.

# De Fourier a Welch

La transformada de Fourier expresa una señal mediante sus componentes de
frecuencia. Para señales digitales y finitas se utiliza la transformada
discreta de Fourier (DFT). La FFT no es una transformada diferente, sino un
algoritmo eficiente para calcular la DFT.

Una FFT devuelve coeficientes complejos que contienen magnitud y fase. No
devuelve directamente una PSD. El periodograma obtiene potencia a partir del
módulo cuadrado de esos coeficientes e incorpora la normalización
correspondiente. Welch reduce la variabilidad de la estimación promediando
periodogramas modificados.

En este proyecto cada época enviada a Welch mide exactamente 2 s. Como se
preparan épocas nuevas cada segundo, la matriz completa presenta un
solapamiento temporal efectivo de 1 s.

Dentro de cada llamada:

- `nperseg = 2 × fs = 256` muestras;
- `noverlap = 0`, porque cada fila ya contiene una única época completa;
- `nfft = fs / 0.5 = 256`, que produce una separación de 0,5 Hz;
- `detrend="constant"`, que resta la media de cada época;
- `window="hann"`, que reduce la fuga espectral en sus bordes.

El solapamiento no se introduce mediante `noverlap`, sino al construir
externamente épocas que comienzan cada 128 muestras.

## Ventana Hann y detrend

La FFT interpreta cada fragmento finito como parte de una señal periódica. Si
el inicio y el final de la época no enlazan, aparece una discontinuidad que
distribuye energía artificial entre frecuencias vecinas. La ventana Hann
reduce progresivamente la amplitud en los bordes y limita esta fuga espectral.

`detrend="constant"` resta el valor medio de cada época. Así se elimina su
componente continua local sin cambiar la unidad de la señal, que sigue siendo
\(\mu V\) antes del cálculo espectral.

La ventana Hann es adimensional. Tanto la corrección de media como la ventana
forman parte del periodograma modificado calculado por Welch.

# Configuración espectral

La configuración seleccionada es:

| Parámetro | Valor |
|:---|:---|
| Ventana | 2 s |
| Avance | 1 s |
| Ventana de ponderación | Hann |
| Detrend | Constante |
| Frecuencia mínima | 0,5 Hz |
| Frecuencia máxima | 30 Hz |
| Separación | 0,5 Hz |
| Escalado | Densidad espectral de potencia |
| Referencia temporal | Centro de la época |

Con `scaling="density"`, Welch devuelve una PSD en
\(\mu V^2/Hz\). La opción `spectrum` devolvería potencia espectral por bin,
pero no es la utilizada en el flujo final.

In [ ]:
PARAMETROS_RECONSTRUCCION = {
    "ventana_welch_s": 2,
    "paso_welch_s": 1,
    "fmin": 0.5,
    "fmax": 30.0,
    "paso_frecuencia": 0.5,
    "modo_welch": "densidad",
    "tiempo_referencia": "centro",
    "umbral_sqi": 15,
    "umbral_ceros": 0.9,
    "referencia_amplitud_uv_rms": 0.0001,
    "orden_filtro_pasa_altos": 1,
    "shift_unilateral_s": 10,
    "shift_bilateral_s": 6,
}

# Unidades y conversión a decibelios

El flujo de unidades es:

1. EEG temporal: \(\mu V\).
2. Welch con `density`: \(\mu V^2/Hz\).
3. Integración aproximada de cada bin de 0,5 Hz:

$$
P_{\mathrm{bin}}(t,f)
=PSD_{\mathrm{Welch}}(t,f)\cdot 0,5\ Hz,
$$

por lo que \(P_{\mathrm{bin}}\) queda expresada en \(\mu V^2\).

4. Conversión a dB:

$$
D(t,f)=10\log_{10}
\left(
\frac{P_{\mathrm{bin}}(t,f)+\varepsilon}
{(0,0001\ \mu V)^2}
\right).
$$

`0,0001 µV RMS` es una referencia de **amplitud**, no
`0,0001 µV²`. Al comparar potencias debe elevarse al cuadrado:

$$
(0,0001\ \mu V)^2=10^{-8}\ \mu V^2.
$$

El término \(\varepsilon\) solo evita `log10(0)`.

El `.f_a` almacena los valores con un factor de escala de 100, por lo que se
divide entre 100 antes de compararlo con la reconstrucción.

# Adaptación a la línea temporal del `.spa`

Una época de 2 s iniciada en \(t=0\) y referida a su centro se asigna a
\(t=1\). Por ello, no existe un espectro centrado en el primer segundo de la
sesión.

Se introduce una fila inicial de `NaN` y las marcas relativas se convierten en
fechas sumándolas al primer instante del `.spa`. Después:

- se eliminan marcas temporales duplicadas conservando la última;
- la DSA se reindexa con todos los segundos oficiales del `.spa`;
- los segundos ausentes permanecen como `NaN`;
- no se prolonga artificialmente la matriz para rellenar huecos.

`frecuencias_c1` y una versión convertida a `float` contienen la misma
información si las columnas ya son numéricas. La conversión solo es necesaria
cuando las etiquetas se han leído como texto.

# Máscara de calidad

La máscara base combina información del `.spa`, de la matriz reconstruida y
de la señal cruda. Una fila se invalida cuando:

- `SQI < 15`;
- `TOTPOW` está ausente;
- existe una discontinuidad superior a un segundo;
- la fila de DSA está completamente vacía;
- más del 90 % de sus valores son cero;
- el segundo crudo está incompleto o contiene al menos un 90 % de ceros.

La pérdida de paquetes puede aparecer como muestras crudas iguales a cero.
Una época Welch dura 2 s, por lo que una pérdida puede afectar a todas las
épocas que se solapan con ese segundo. La máscara raw se proyecta sobre esas
ventanas antes de alinearla con el `.spa`.

La máscara ARTF2, basada en indicadores adicionales del archivo `.ara`, se
probó experimentalmente pero se descartó como máscara obligatoria porque
eliminaba aproximadamente un tercio de la información sin mejorar de forma
consistente las métricas.

# Suavizado y shift

Los segundos inválidos se convierten en `NaN` **antes** del suavizado. La media
móvil causal ignora esos valores y utiliza únicamente los segundos válidos
disponibles dentro de la ventana indicada por `SpSmooth`:

```python
trabajo.loc[mask_base, :] = np.nan
dsa_suavizada = trabajo.rolling(
    window=SpSmooth,
    min_periods=1,
    center=False,
).mean()
```

No se concatenan los segundos válidos como si fueran temporalmente
consecutivos: la ventana sigue recorriendo la línea temporal original. Los
`NaN` simplemente no contribuyen al denominador de la media.

Después se desplaza la DSA reconstruida:

- 10 s en unilateral;
- 6 s en bilateral.

Un shift positivo coloca en \(t+s\) el valor reconstruido en \(t\). No cambia
la potencia ni las frecuencias; modifica su asignación temporal. Finalmente se
reaplica la máscara base y también se enmascaran los bordes sin información
creados por el desplazamiento.

# Variables superpuestas y visualización

La aplicación no sustituye el SEF y la MEF del `.spa` por valores recalculados
desde la reconstrucción. Las curvas mostradas proceden de:

- `SEF08`;
- `MEDFRQ08`;
- `DB13U01` para el índice BIS;
- `ASYM09` en bilateral.

Estas variables se alinean con la misma timeline y reciben la máscara de la
DSA correspondiente. En bilateral, `ASYM09` se oculta si falta información en
cualquiera de los hemisferios.

Calcular SEF y MEF propios sobre potencia lineal puede ser útil como
experimento de validación, pero no forma parte de la visualización principal.

La escala cromática es lineal y fija entre 49 y 94 dB. No se utiliza
`gamma=0.25`, percentiles ni una normalización independiente por registro. Los
valores exteriores al intervalo se saturan visualmente en los extremos, pero
no se eliminan de la matriz.

# Validación y alternativas

La reconstrucción se compara con el `.f_a` sobre celdas válidas comunes:

- Pearson: semejanza del patrón espectral;
- MAE y RMSE: error absoluto en dB;
- bias: desplazamiento medio en dB.

No se aplica z-score antes de MAE o RMSE porque eliminaría diferencias de nivel
y contraste y las métricas dejarían de estar expresadas en dB.

Resultados provisionales de la configuración seleccionada:

| Método | Suavizado | Máscara | Pearson medio | MAE | RMSE |
|:---|:---|:---|---:|---:|---:|
| Welch | `rolling` | Base | 0,9149 | 1,816 dB | 2,581 dB |

`spectrogram`, configurado con los mismos parámetros, fue numéricamente
equivalente a Welch. Las wavelets de Morlet produjeron peores métricas frente
al `.f_a`. Esto no significa que sean inadecuadas para detectar transitorios,
sino que no aportaron ventaja para reproducir esta DSA de referencia.

# Resumen del flujo final

1. Detectar modalidad por `.r2a` o `.r4a`.
2. Leer cabecera, inicio temporal, EEG crudo y `.spa`.
3. Convertir las muestras a \(\mu V\) con pendiente y offset.
4. Ajustar el raw al intervalo temporal del `.spa`.
5. Seleccionar C1 en unilateral y C1/C3 en bilateral.
6. Aplicar el pasa-altos causal indicado por `LoFilter`.
7. Calcular Welch con ventanas de 2 s y avance de 1 s.
8. Conservar 0,5--30 Hz en bins de 0,5 Hz.
9. Integrar la PSD por bin y convertirla a dB.
10. Insertar la fila inicial vacía y reindexar con la timeline del `.spa`.
11. Construir la máscara base.
12. Excluir inválidos, aplicar `rolling` según `SpSmooth` y realizar el shift.
13. Reaplicar la máscara a DSA y parámetros.
14. Representar con una escala fija de 49--94 dB.
15. Validar frente al `.f_a` sin z-score.

# Distribución recomendada en la memoria

Con un límite aproximado de 50 páginas, no conviene trasladar literalmente
todo este cuaderno.

| Contenido | Ubicación recomendada |
|:---|:---|
| Objetivo y alcance de la reconstrucción | Metodología |
| Archivos, canales y alineación con `.spa` | Metodología |
| Filtro, Welch, máscara, suavizado y shift finales | Metodología |
| Fórmula de conversión a dB | Metodología |
| Conceptos generales de FFT, PSD, Hann y fuga espectral | Marco teórico, de forma breve |
| Derivaciones completas de FT/DFT/FFT | Omitir del PDF o conservar solo en el notebook |
| Comparación Welch, `spectrogram` y wavelets | Apéndice G |
| Comparación `rolling` y `ewm` | Apéndice G |
| Máscara base frente a ARTF2 | Apéndice G |
| Fórmulas e interpretación detallada de métricas | Apéndice G |
| Resultados por registro y limitaciones | Apéndice G |
| Código y nombres de funciones | Manual del programador o repositorio |

La memoria debe explicar **qué procedimiento se utilizó**. El apéndice G debe
demostrar **por qué se eligió frente a las alternativas**.